# Neural Networks from Scratch

## 1. Micrograd

In [ ]:
import math
import random

class Value:
  def __init__(self, data, _children=(), _op='', label=''):
    self.data = data
    self._prev = set(_children)
    self._op = _op
    self.label = label
    self._backward = lambda: None
    self.grad = 0.0

  def __repr__(self):
    return f"Value(data={self.data})"

  def __add__(self, other):
    other = other if isinstance(other, Value) else Value(other)
    out = Value(self.data + other.data, (self, other), '+')
    def _backward():
      self.grad += 1.0 * out.grad
      other.grad += 1.0 * out.grad
    out._backward = _backward
    return out

  def __mul__(self, other):
    other = other if isinstance(other, Value) else Value(other)
    out = Value(self.data * other.data, (self, other), '*')
    def _backward():
      self.grad += other.data * out.grad
      other.grad += self.data * out.grad
    out._backward = _backward
    return out

  def __pow__(self, other):
    assert isinstance(other, (int, float))
    out = Value(self.data ** other, (self,), f'**{other}')
    def _backward():
      self.grad += other * self.data ** (other - 1) * out.grad
    out._backward = _backward
    return out

  def __radd__(self, other):
    return self + other

  def __rmul__(self, other):
    return self * other

  def __truediv__(self, other):
    return self * other ** -1

  def __neg__(self):
    return self * -1

  def __sub__(self, other):
    return self + (-other)

  def __rsub__(self, other):
    return other + (-self)

  def tanh(self):
    x = self.data
    t = (math.exp(2 * x) - 1) / (math.exp(2 * x) + 1)
    out = Value(t, (self,), 'tanh')
    def _backward():
      self.grad += (1 - t ** 2) * out.grad
    out._backward = _backward
    return out

  def exp(self):
    out = Value(math.exp(self.data), (self,), 'exp')
    def _backward():
      self.grad += out.data * out.grad
    out._backward = _backward
    return out

  def backward(self):
    topo = []
    visited = set()
    def build_topo(v):
      if v not in visited:
        visited.add(v)
        for child in v._prev:
          build_topo(child)
        topo.append(v)
    build_topo(self)
    self.grad = 1.0
    for node in reversed(topo):
      node._backward()

In [ ]:
class Neuron:
  def __init__(self, nin):
    self.w = [Value(random.uniform(-1, 1)) for _ in range(nin)]
    self.b = Value(random.uniform(-1, 1))

  def __call__(self, x):
    act = sum((wi * xi for wi, xi in zip(self.w, x)), self.b)
    return act.tanh()

  def parameters(self):
    return self.w + [self.b]


class Layer:
  def __init__(self, nin, nout):
    self.neurons = [Neuron(nin) for _ in range(nout)]

  def __call__(self, x):
    outs = [n(x) for n in self.neurons]
    return outs[0] if len(outs) == 1 else outs

  def parameters(self):
    return [p for neuron in self.neurons for p in neuron.parameters()]


class MLP:
  def __init__(self, nin, nouts):
    sz = [nin] + nouts
    self.layers = [Layer(sz[i], sz[i + 1]) for i in range(len(nouts))]

  def __call__(self, x):
    for layer in self.layers:
      x = layer(x)
    return x

  def parameters(self):
    return [p for layer in self.layers for p in layer.parameters()]

In [ ]:
xs = [
  [2.0, 3.0, -1.0],
  [3.0, -1.0, 0.5],
  [0.5, 1.0, 1.0],
  [1.0, 1.0, -1.0],
]
ys = [1.0, -1.0, -1.0, 1.0]

net = MLP(3, [4, 4, 1])

for k in range(50):
  ypred = [net(x) for x in xs]
  loss = sum((yout - ygt) ** 2 for ygt, yout in zip(ys, ypred))

  for p in net.parameters():
    p.grad = 0.0
  loss.backward()

  for p in net.parameters():
    p.data += -0.11 * p.grad

  print(k, loss.data)

print(ypred)

## 2. Data

In [ ]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
%matplotlib inline

!wget -nc https://raw.githubusercontent.com/karpathy/makemore/master/names.txt

names = open('names.txt', 'r').read().splitlines()
print(len(names))

In [ ]:
block_size = 3
chars = sorted(list(set(''.join(names))))
vocab_size = len(chars) + 1
stoi = {s: i + 1 for i, s in enumerate(chars)}
stoi['.'] = 0
itos = {i: s for s, i in stoi.items()}

def build_dataset(words):
  X, Y = [], []
  for w in words:
    context = [0] * block_size
    for ch in w + '.':
      ix = stoi[ch]
      X.append(context)
      Y.append(ix)
      context = context[1:] + [ix]
  return torch.tensor(X), torch.tensor(Y)

random.seed(42)
shuffled = list(names)
random.shuffle(shuffled)
n1 = int(0.8 * len(shuffled))
n2 = int(0.9 * len(shuffled))

Xtr, Ytr = build_dataset(shuffled[:n1])
Xdev, Ydev = build_dataset(shuffled[n1:n2])
Xte, Yte = build_dataset(shuffled[n2:])
print(Xtr.shape, Xdev.shape, Xte.shape)

## 3. MLP with manual BatchNorm

In [ ]:
n_embd = 10
n_hidden = 300

g = torch.Generator().manual_seed(2147483647)
C = torch.randn((vocab_size, n_embd), generator=g)
W1 = torch.randn((n_embd * block_size, n_hidden), generator=g) * 0.01
W2 = torch.randn((n_hidden, vocab_size), generator=g) * 0.01
b2 = torch.zeros(vocab_size)
bngain = torch.ones((1, n_hidden))
bnbias = torch.zeros((1, n_hidden))
bnmean_running = torch.zeros((1, n_hidden))
bnstd_running = torch.ones((1, n_hidden))

parameters = [C, W1, W2, b2, bngain, bnbias]
for p in parameters:
  p.requires_grad = True
print(sum(p.nelement() for p in parameters))

In [ ]:
@torch.no_grad()
def split_loss_mlp(X, Y):
  emb = C[X]
  embcat = emb.view(emb.shape[0], -1)
  hpreact = embcat @ W1
  bnmeani = hpreact.mean(0, keepdim=True)
  bnstdi = hpreact.std(0, keepdim=True)
  hpreact = bngain * (hpreact - bnmeani) / bnstdi + bnbias
  h = torch.tanh(hpreact)
  logits = h @ W2 + b2
  return F.cross_entropy(logits, Y).item()

In [ ]:
lr = 0.1
best_dev = float('inf')
bad_checks = 0
stepi, lossi, devi = [], [], []

for i in range(60000):
  bi = torch.randint(0, Xtr.shape[0], (32,))
  emb = C[Xtr[bi]]
  embcat = emb.view(emb.shape[0], -1)
  hpreact = embcat @ W1
  bnmeani = hpreact.mean(0, keepdim=True)
  bnstdi = hpreact.std(0, keepdim=True)
  hpreact = bngain * (hpreact - bnmeani) / bnstdi + bnbias

  with torch.no_grad():
    bnmean_running = 0.999 * bnmean_running + 0.001 * bnmeani
    bnstd_running = 0.999 * bnstd_running + 0.001 * bnstdi

  h = torch.tanh(hpreact)
  logits = h @ W2 + b2
  loss = F.cross_entropy(logits, Ytr[bi])

  for p in parameters:
    p.grad = None
  loss.backward()
  for p in parameters:
    p.data += -lr * p.grad

  stepi.append(i)
  lossi.append(loss.item())

  if i % 2000 == 0 and i > 0:
    dev_loss = split_loss_mlp(Xdev, Ydev)
    devi.append(dev_loss)
    print(i, 'lr', lr, 'dev', round(dev_loss, 4))
    if dev_loss < best_dev - 0.001:
      best_dev = dev_loss
      bad_checks = 0
    else:
      bad_checks += 1
      if bad_checks >= 2:
        lr = lr / 3
        bad_checks = 0
        if lr < 0.001:
          break

print('train', split_loss_mlp(Xtr, Ytr))
print('dev', split_loss_mlp(Xdev, Ydev))
print('test', split_loss_mlp(Xte, Yte))

In [ ]:
plt.plot(stepi, lossi)

In [ ]:
g = torch.Generator().manual_seed(2147483647 + 10)

for _ in range(20):
  out = []
  context = [0] * block_size
  while True:
    emb = C[torch.tensor([context])]
    embcat = emb.view(1, -1)
    hpreact = embcat @ W1
    hpreact = bngain * (hpreact - bnmean_running) / bnstd_running + bnbias
    h = torch.tanh(hpreact)
    logits = h @ W2 + b2
    probs = F.softmax(logits, dim=1)
    ix = torch.multinomial(probs, num_samples=1, generator=g).item()
    context = context[1:] + [ix]
    out.append(ix)
    if ix == 0:
      break
  print(''.join(itos[i] for i in out))

## 4. Layers: Linear, Tanh, BatchNorm1d

In [ ]:
class Linear:
  def __init__(self, fan_in, fan_out, bias=True):
    self.weight = torch.randn((fan_in, fan_out), generator=g) / fan_in ** 0.5
    self.bias = torch.zeros(fan_out) if bias else None

  def __call__(self, X):
    self.out = X @ self.weight
    if self.bias is not None:
      self.out += self.bias
    return self.out

  def parameters(self):
    return [self.weight] + ([self.bias] if self.bias is not None else [])


class Tanh:
  def __call__(self, X):
    self.out = torch.tanh(X)
    return self.out

  def parameters(self):
    return []


class BatchNorm1d:
  def __init__(self, dim, eps=1e-5, momentum=0.1):
    self.eps = eps
    self.momentum = momentum
    self.training = True
    self.gamma = torch.ones(dim)
    self.beta = torch.zeros(dim)
    self.running_mean = torch.zeros(dim)
    self.running_var = torch.ones(dim)

  def __call__(self, X):
    if self.training:
      xmean = X.mean(0, keepdim=True)
      xvar = X.var(0, keepdim=True)
    else:
      xmean = self.running_mean
      xvar = self.running_var
    xhat = (X - xmean) / torch.sqrt(xvar + self.eps)
    self.out = xhat * self.gamma + self.beta
    if self.training:
      with torch.no_grad():
        self.running_mean = (1 - self.momentum) * self.running_mean + self.momentum * xmean
        self.running_var = (1 - self.momentum) * self.running_var + self.momentum * xvar
    return self.out

  def parameters(self):
    return [self.gamma, self.beta]

## 5. Deep network

In [ ]:
n_embd = 10
n_hidden = 100

g = torch.Generator().manual_seed(2147483647)
C = torch.randn((vocab_size, n_embd), generator=g)

layers = []
for i in range(6):
  if i == 0:
    layers.extend([Linear(n_embd * block_size, n_hidden, bias=False), BatchNorm1d(n_hidden), Tanh()])
  elif i == 5:
    layers.extend([Linear(n_hidden, vocab_size, bias=False), BatchNorm1d(vocab_size)])
  else:
    layers.extend([Linear(n_hidden, n_hidden, bias=False), BatchNorm1d(n_hidden), Tanh()])

with torch.no_grad():
  layers[-1].gamma *= 0.1

parameters = [C] + [p for layer in layers for p in layer.parameters()]
for p in parameters:
  p.requires_grad = True
print(sum(p.nelement() for p in parameters))

In [ ]:
max_steps = 15000
lossi = []

for i in range(max_steps):
  ix = torch.randint(0, Xtr.shape[0], (32,), generator=g)
  Xb, Yb = Xtr[ix], Ytr[ix]

  x = C[Xb].view(Xb.shape[0], -1)
  for layer in layers:
    x = layer(x)
  loss = F.cross_entropy(x, Yb)

  for p in parameters:
    p.grad = None
  loss.backward()

  lr = 0.1 if i < 10000 else 0.01
  for p in parameters:
    p.data += -lr * p.grad

  lossi.append(loss.item())
  if i % 5000 == 0:
    print(f'{i:7d}/{max_steps:7d}: {loss.item():.4f}')

print(lossi[0], lossi[-1])

In [ ]:
for layer in layers:
  layer.training = False

@torch.no_grad()
def split_loss(X, Y):
  x = C[X].view(X.shape[0], -1)
  for layer in layers:
    x = layer(x)
  return F.cross_entropy(x, Y).item()

print('train', split_loss(Xtr, Ytr))
print('dev', split_loss(Xdev, Ydev))
print('test', split_loss(Xte, Yte))

In [ ]:
g = torch.Generator().manual_seed(2147483647 + 10)

for _ in range(20):
  out = []
  context = [0] * block_size
  while True:
    x = C[torch.tensor([context])].view(1, -1)
    for layer in layers:
      x = layer(x)
    probs = F.softmax(x, dim=1)
    ix = torch.multinomial(probs, num_samples=1, generator=g).item()
    context = context[1:] + [ix]
    out.append(ix)
    if ix == 0:
      break
  print(''.join(itos[i] for i in out))